In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## 必要なライブラリのインポート

In [ ]:
import pandas as pd # データ操作と分析のためのライブラリ
import numpy as np # 数値計算のためのライブラリ（今回は直接使用しないが、データ分析の定番）
import matplotlib.pyplot as plt # グラフ描画のためのライブラリ
import seaborn as sns # 統計的グラフ描画のためのライブラリ（matplotlibをより美しく使いやすくする）
import re # 正規表現操作のためのライブラリ（テキスト前処理に使用）
from collections import Counter # リスト内の要素の出現回数をカウントするためのクラス
from wordcloud import WordCloud # ワードクラウドを生成するためのライブラリ

# グラフのスタイルをSeabornのホワイトグリッドに設定
sns.set_style("whitegrid")
# Jupyter NotebookやColabでグラフをインライン表示するためのマジックコマンド
%matplotlib inline

## データセットの読み込み

In [ ]:
# データセットのパスを設定（Kaggle Notebooks環境を想定）
train_path = '/kaggle/input/nlp-getting-started/train.csv'
test_path = '/kaggle/input/nlp-getting-started/test.csv'
sample_submission_path = '/kaggle/input/nlp-getting-started/sample_submission.csv'

# CSVファイルをPandas DataFrameとして読み込む
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_submission_df = pd.read_csv(sample_submission_path)

# 読み込んだtrain_dfの最初の5行を表示（Jupyter/Colabで自動表示される）
train_df.head()

In [ ]:
# test_dfの最初の5行を表示
test_df.head()

In [ ]:
# sample_submission_dfの最初の5行を表示
sample_submission_df.head()

## データの概要把握

In [ ]:
# train_dfの列情報、非Null値の数、データ型などを表示
train_df.info()

In [ ]:
# train_dfの各列における欠損値の数を表示
train_df.isnull().sum()

In [ ]:
# test_dfの列情報、非Null値の数、データ型などを表示
test_df.info()

In [ ]:
# test_dfの各列における欠損値の数を表示
test_df.isnull().sum()

## ターゲット変数の分布確認

In [ ]:
# target列の値の出現回数をカウントし、比率も算出
# train_df['target'].value_counts()
# train_df['target'].value_counts(normalize=True) * 100

# target列の分布を棒グラフで可視化
plt.figure(figsize=(6, 4)) # グラフのサイズを設定
sns.countplot(x='target', data=train_df, palette='viridis') # target列の各値の出現回数を棒グラフで表示
plt.title('Distribution of Target Variable') # グラフのタイトルを設定
plt.xlabel('Target (0: Not Disaster, 1: Disaster)') # x軸のラベルを設定
plt.ylabel('Count') # y軸のラベルを設定
plt.show() # グラフを表示

## テキストデータの探索

In [ ]:
# 各ツイートの文字数を計算し、新しい列'text_length'としてtrain_dfに追加
train_df['text_length'] = train_df['text'].apply(len)
# 各ツイートの文字数を計算し、新しい列'text_length'としてtest_dfに追加
test_df['text_length'] = test_df['text'].apply(len)

# 各ツイートの単語数を計算し、新しい列'word_count'としてtrain_dfに追加
# textをスペースで分割し、その要素数を数える
train_df['word_count'] = train_df['text'].apply(lambda x: len(x.split()))
# 各ツイートの単語数を計算し、新しい列'word_count'としてtest_dfに追加
test_df['word_count'] = test_df['text'].apply(lambda x: len(x.split()))

plt.figure(figsize=(12, 5)) # グラフ全体のサイズを設定

# 文字数分布のヒストグラム
plt.subplot(1, 2, 1) # 1行2列のグラフの1番目の位置に配置
sns.histplot(train_df['text_length'], bins=50, kde=True, color='skyblue') # 文字数でヒストグラムとKDE（カーネル密度推定）をプロット
plt.title('Distribution of Tweet Length (Characters)') # サブグラフのタイトル
plt.xlabel('Number of Characters') # x軸のラベル
plt.ylabel('Count') # y軸のラベル

# 単語数分布のヒストグラム
plt.subplot(1, 2, 2) # 1行2列のグラフの2番目の位置に配置
sns.histplot(train_df['word_count'], bins=50, kde=True, color='lightcoral') # 単語数でヒストグラムとKDEをプロット
plt.title('Distribution of Tweet Length (Words)') # サブグラフのタイトル
plt.xlabel('Number of Words') # x軸のラベル
plt.ylabel('Count') # y軸のラベル

plt.tight_layout() # サブグラフ間のスペースを自動調整
plt.show() # グラフを表示

# train_dfのテキスト長さと単語数の基本的な統計量（平均、標準偏差、最小、最大など）を表示
train_df[['text_length', 'word_count']].describe()

In [ ]:
# テキストから単語を抽出するための関数を定義
def get_all_words(text_series):
    all_words = [] # 全ての単語を格納するリスト
    # テキストから句読点を除去し、全て小文字に変換（正規表現を使用）
    cleaned_text = text_series.str.replace(r'[^\w\s]', '', regex=True).str.lower()
    for text in cleaned_text:
        # クリーニングされたテキストをスペースで分割し、単語リストに追加
        all_words.extend(text.split())
    return all_words

# 災害ツイート（targetが1の行）から単語を抽出
disaster_words = get_all_words(train_df[train_df['target'] == 1]['text'])
# 非災害ツイート（targetが0の行）から単語を抽出
non_disaster_words = get_all_words(train_df[train_df['target'] == 0]['text'])

# 災害ツイート内の単語の出現回数をカウント
disaster_word_counts = Counter(disaster_words)
# 非災害ツイート内の単語の出現回数をカウント
non_disaster_word_counts = Counter(non_disaster_words)

# 災害ツイートで最も頻繁に出現する単語のトップ10を表示
# disaster_word_counts.most_common(10)

# 非災害ツイートで最も頻繁に出現する単語のトップ10を表示
# non_disaster_word_counts.most_common(10)

# ワードクラウドを生成するためのフィギュアとサブプロットを設定
plt.figure(figsize=(15, 7))

# 災害ツイートのワードクラウドを生成し表示
plt.subplot(1, 2, 1) # 1行2列のグラフの1番目の位置に配置
# WordCloudオブジェクトを生成。背景色を白に設定。
wordcloud_disaster = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(disaster_word_counts)
plt.imshow(wordcloud_disaster, interpolation='bilinear') # ワードクラウドを表示
plt.axis('off') # 軸を非表示にする
plt.title('Word Cloud for Disaster Tweets') # サブグラフのタイトル

# 非災害ツイートのワードクラウドを生成し表示
plt.subplot(1, 2, 2) # 1行2列のグラフの2番目の位置に配置
# WordCloudオブジェクトを生成。背景色を白に設定。
wordcloud_non_disaster = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(non_disaster_word_counts)
plt.imshow(wordcloud_non_disaster, interpolation='bilinear') # ワードクラウドを表示
plt.axis('off') # 軸を非表示にする
plt.title('Word Cloud for Non-Disaster Tweets') # サブグラフのタイトル

plt.tight_layout() # サブグラフ間のスペースを自動調整
plt.show() # グラフを表示

## keywordとlocation列の探索

In [ ]:
# train_dfの'keyword'列で最も頻繁に出現する値のトップ10を表示
# train_df['keyword'].value_counts().head(10)

# train_dfの'location'列で最も頻繁に出現する値のトップ10を表示
# train_df['location'].value_counts().head(10)

# 'keyword'列と'target'列の関係を可視化
plt.figure(figsize=(12, 6)) # グラフのサイズを設定
# 'keyword'をY軸に、'target'をhue（色分け）に設定し、カウントプロットを作成
# 最も頻出するトップ10のキーワードに絞って表示
sns.countplot(y='keyword', hue='target', data=train_df,
              order=train_df['keyword'].value_counts().index[:10],
              palette='coolwarm') # カラーパレットを設定
plt.title('Top 10 Keywords by Target') # グラフのタイトルを設定
plt.xlabel('Count') # x軸のラベルを設定
plt.ylabel('Keyword') # y軸のラベルを設定
plt.show() # グラフを表示

## データの前処理

In [ ]:
# train_dfとtest_dfのkeyword列の欠損値を「no_keyword」で補完
train_df['keyword'].fillna('no_keyword', inplace=True)
test_df['keyword'].fillna('no_keyword', inplace=True)

# train_dfとtest_dfのlocation列の欠損値を「no_location」で補完
train_df['location'].fillna('no_location', inplace=True)
test_df['location'].fillna('no_location', inplace=True)

train_df.isnull().sum()
test_df.isnull().sum()

In [ ]:
# テキストクリーニング用の関数を定義
def clean_text(text):
    # URLを除去
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # HTMLタグを除去
    text = re.sub(r'<.*?>', '', text)
    # 絵文字を除去（広範囲な絵文字に対応）
    emoji_pattern = re.compile(
        '['
        '\U0001F600-\U0001F64F'  # Emoticons
        '\U0001F300-\U0001F5FF'  # Symbols & Pictographs
        '\U0001F680-\U0001F6FF'  # Transport & Map Symbols
        '\U0001F1E0-\U0001F1FF'  # Flags (iOS)
        '\U00002702-\U000027B0'
        '\U000024C2-\U0001F251'
        ']+',
        flags=re.UNICODE
    )
    text = emoji_pattern.sub(r'', text)
    # 句読点を除去
    text = re.sub(r'[^\w\s]', '', text)
    # 数字を除去
    text = re.sub(r'\d+', '', text)
    # 小文字に変換
    text = text.lower()
    return text

# train_dfとtest_dfのtext列にクリーニング関数を適用
train_df['cleaned_text'] = train_df['text'].apply(clean_text)
test_df['cleaned_text'] = test_df['text'].apply(clean_text)

# クリーニング後のテキストの最初の5行を確認
train_df[['text', 'cleaned_text']].head()

In [ ]:
import nltk # 自然言語処理ライブラリ
from nltk.corpus import stopwords # ストップワードのリスト

# NLTKのストップワードリストをダウンロード（初回のみ必要）
# nltk.download('stopwords')

# 英語のストップワードリストを取得
stop_words = set(stopwords.words('english'))

# ストップワード除去用の関数を定義
def remove_stopwords(text):
    # テキストを単語に分割
    words = text.split()
    # ストップワードでない単語のみを抽出して結合
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)

# クリーニング済みテキストにストップワード除去関数を適用
train_df['processed_text'] = train_df['cleaned_text'].apply(remove_stopwords)
test_df['processed_text'] = test_df['cleaned_text'].apply(remove_stopwords)

# 処理後のテキストの最初の5行を確認
train_df[['cleaned_text', 'processed_text']].head()

## 特徴量エンジニアリング

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer # TF-IDF特徴量変換のためのクラス

# TF-IDFベクトライザーのインスタンスを作成
# min_df=5: 少なくとも5つの文書に出現する単語のみを考慮
# max_df=0.8: 文書の80%以上に出現する単語（一般的な単語）は無視
# max_features=10000: 最大10000個の特徴量に制限
tfidf_vectorizer = TfidfVectorizer(min_df=5, max_df=0.8, max_features=10000)

# train_dfとtest_dfのprocessed_text列を結合してTF-IDFを学習
# 訓練データとテストデータ全体で単語辞書を構築することで、一貫した特徴量が得られる
full_text = pd.concat([train_df['processed_text'], test_df['processed_text']])
tfidf_vectorizer.fit(full_text)

# 訓練データとテストデータにTF-IDF変換を適用
X_train_tfidf = tfidf_vectorizer.transform(train_df['processed_text'])
X_test_tfidf = tfidf_vectorizer.transform(test_df['processed_text'])

# ターゲット変数（ラベル）をY_trainに格納
y_train = train_df['target']

# TF-IDF特徴量行列の形状を確認（行数、特徴量数）
X_train_tfidf.shape
X_test_tfidf.shape

## モデル構築と学習

In [ ]:
from sklearn.linear_model import LogisticRegression # ロジスティック回帰モデル
from sklearn.model_selection import train_test_split # データを訓練用と検証用に分割
from sklearn.metrics import f1_score # F1スコアの計算

# 訓練データをさらに訓練セットと検証セットに分割（モデルの性能評価のため）
# stratified=Trueでtargetの割合が訓練・検証セットで維持されるように分割
X_train, X_val, y_train_split, y_val = train_test_split(
    X_train_tfidf, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# ロジスティック回帰モデルのインスタンスを作成
# max_iterを増やして収束しやすくする
model = LogisticRegression(max_iter=1000, random_state=42)

# モデルを訓練データで学習
model.fit(X_train, y_train_split)

# 検証データで予測
y_val_pred = model.predict(X_val)

# 検証データでのF1スコアを計算し表示
f1 = f1_score(y_val, y_val_pred)
f1

## 予測と提出ファイルの作成

In [ ]:
# テストデータに対して予測を実行
final_predictions = model.predict(X_test_tfidf)

# 提出用DataFrameを作成
# sample_submission_dfの'target'列を予測結果で更新
submission_df = sample_submission_df.copy()
submission_df['target'] = final_predictions

# 提出ファイルの最初の5行を確認
submission_df.head()

In [ ]:
# 提出ファイルをCSV形式で保存
# index=Falseを指定して、DataFrameのインデックスがCSVに含まれないようにする
submission_df.to_csv('submission.csv', index=False)